In [3]:
#pip install langchain-ollama

In [4]:
#pip install langchain-community


In [1]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

C:\Users\Jawad\AppData\Local\Temp\ipykernel_32468\704489491.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
c:\Users\Jawad\Documents\MASK AI Internship\Document Semantic Search System\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PDF_DIR=r"C:\Users\Jawad\Documents\MASK AI Internship\Document-Based AI Question Answering System\data\raw"
CHROMA_DIR="./chroma_db"

In [3]:
import torch
print(torch.__version__)

2.13.0+cpu


In [3]:
emb=HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings":True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3452.29it/s]


In [4]:
def build_db():
    docs = []

    pdf_files = list(Path(PDF_DIR).glob("*.pdf"))
    print(f"Found PDFs: {len(pdf_files)}")

    for pdf in pdf_files:
        docs.extend(PyPDFLoader(str(pdf)).load())

    print(f"Pages loaded: {len(docs)}")

    if not docs:
        raise ValueError("No PDF documents found.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(docs)

    print(f"Chunks created: {len(chunks)}")

    if not chunks:
        raise ValueError("No chunks were created.")

    print("Testing embedding...")
    vec = emb.embed_query("test")
    print("Embedding dimension:", len(vec))

    db = Chroma.from_documents(
        documents=chunks,
        embedding=emb,
        persist_directory=CHROMA_DIR,
    )

    return db

In [5]:
def load_db():
    return Chroma(persist_directory=CHROMA_DIR,embedding_function=emb)

In [6]:
prompt=ChatPromptTemplate.from_template("""
You are a helpful Questioning Answering assistant.
Use ONLY the context below. If unavailable, say you don't know.

Context:
{context}

Question:
{question}

Answer:
""")


In [7]:
db=load_db() if Path(CHROMA_DIR).exists() else build_db()

In [8]:
llm=ChatOllama(model="qwen2.5:3b",temperature=0) #model="llama3.1:8b"

In [9]:
q="What is Attention"  #input("Question (exit to quit): ")""

In [10]:

results=db.similarity_search_with_relevance_scores(q,k=3)
ctx=[]
print("\nTop 3 chunks:")
for i,(doc,score) in enumerate(results,1):
    print(f"{i}. score={score:.4f} source={doc.metadata.get('source')}")
    print(doc.page_content[:200],"...\n")
    ctx.append(doc.page_content)
msgs=prompt.format_messages(context="\n\n".join(ctx),question=q)
ans=llm.invoke(msgs)
print("Answer:\n",ans.content,"\n")


Top 3 chunks:
1. score=0.5219 source=C:\Users\Jawad\Documents\MASK AI Internship\Document-Based AI Question Answering System\data\raw\attention-is-all-you-need.pdf
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the seque ...

2. score=0.5111 source=C:\Users\Jawad\Documents\MASK AI Internship\Document-Based AI Question Answering System\data\raw\attention-is-all-you-need.pdf
Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
query with all  ...

3. score=0.4845 source=C:\Users\Jawad\Documents\MASK AI Internship\Document-Based AI Question Answering System\data\raw\attention-is-all-you-need.pdf
around each of the sub-layers, followed by layer normalization. We also modify the self-attention
sub-layer in the decoder 